In [ ]:
# FISHER INFORMATION AND GLOBAL ED FOR CL
# ---------------------------------------------------
# HELPERS: set a flat weight vector into the model
# ---------------------------------------------------
def get_shapes_sizes(model):
    ws = model.get_weights()
    shapes = [w.shape for w in ws]
    sizes = [w.size for w in ws]
    return shapes, sizes

def set_flat_weights(model, flat, shapes, sizes):
    new_weights = []
    idx = 0
    for shape, size in zip(shapes, sizes):
        new_weights.append(flat[idx:idx+size].reshape(shape).astype(np.float32))
        idx += size
    model.set_weights(new_weights)
    
def compute_fisher_true(model, x_samples):
    """
    True Fisher Information Matrix for ONE set of weights,
    averaged over x_samples, using predicted class probabilities
    (not the true label y).
    """
    params = model.trainable_variables
    d = np.sum([tf.size(p).numpy() for p in params])

    F = np.zeros((d, d))
    N = len(x_samples)

    for x in x_samples:

        x = tf.convert_to_tensor([x], dtype=tf.float32)

        preds = model(x, training=False)[0]   # shape (num_classes,)
        num_classes = preds.shape[0]

        for c in range(num_classes):

            with tf.GradientTape() as tape:
                preds_c = model(x, training=False)[0]
                log_p_c = tf.math.log(preds_c[c] + 1e-12)

            grad = tape.gradient(log_p_c, params)

            # flatten gradients
            g = np.concatenate([
                tf.reshape(gr, [-1]).numpy()
                for gr in grad
            ])

            p_c = preds[c].numpy()
            F += p_c * np.outer(g, g)

    F /= N
    return F


def effective_dimension_global(fishers, n):
    """
    Global Effective Dimension — many weight samples (fishers is a list/array
    of Fisher matrices, one per theta). Comparable to Qiskit's EffectiveDimension.
    """
    d = fishers[0].shape[0]
    num_thetas = len(fishers)

    # average Fisher across all thetas, used only for the trace normalization
    F_avg = np.mean(fishers, axis=0)
    trace = np.trace(F_avg)

    kappa = n / (2 * np.pi * np.log(n))

    logdets = []
    for F in fishers:
        F_hat = d * F / trace
        M = np.eye(d) + kappa * F_hat
        sign, logdet = np.linalg.slogdet(M)
        logdets.append(0.5 * logdet)

    # logsumexp, done manually
    logdets = np.array(logdets)
    max_val = np.max(logdets)
    log_sum_exp = max_val + np.log(np.sum(np.exp(logdets - max_val)))

    d_eff = 2 * (log_sum_exp - np.log(num_thetas)) / np.log(kappa)
    return d_eff


# ---------------------------------------------------
# RUN: GLOBAL EFFECTIVE DIMENSION FOR THE CLASSICAL MODEL
# ---------------------------------------------------
CL_model = build_model2(input_lenght=4, depth=3, n_class=2)
d = CL_model.count_params()   # 60
m = 4

weights = np.random.uniform(-1, 1, size=(300, d))
inputs  = np.random.normal(0, 1, size=(100, m))

fishers = []
for t in range(weights.shape[0]):
    set_flat_weights(CL_model, weights[t], *get_shapes_sizes(CL_model))  # reuse from earlier
    F_t = compute_fisher_true(CL_model, inputs)
    fishers.append(F_t)
    if t % 50 == 0:
        print(f"done {t}/{weights.shape[0]}")

n_list = [1000, 3000, 5000, 8000, 10000, 40000, 60000, 100000, 150000, 200000, 500000, 1000000]

global_eff_cl_vs_n = [effective_dimension_global(fishers, n) for n in n_list]

for n, eff in zip(n_list, global_eff_cl_vs_n):
    print(f"n={n:>8}: Global ED CL = {eff:.4f}  normalized: {eff/d:.4f}")

with open("global_eff_cl_syn.pkl", "wb") as f:
    pickle.dump(global_eff_cl_vs_n, f)